# 🫀 ECG Model — One-Time Setup (No Kaggle, No gdown, No HuggingFace)

## How it works
```
wfdb.dl_database('mitdb')           ← downloads raw .dat/.hea files from PhysioNet
        │                              (official source, always works, free)
        ▼
GQRS R-peak detector                ← finds heartbeat positions
        ▼
Beat segmentation                   ← cuts 187-sample windows around each beat
        ▼
Train 1D-ResNet  (~4 min on T4)     ← learns to classify arrhythmia type
        ▼
Save weights → Google Drive         ← permanent storage
```

## Run this notebook ONCE → use the weights forever in your main notebook

## 5 Classes (AAMI standard)
| ID | Symbol | Name |
|----|--------|------|
| 0 | N | Normal beat |
| 1 | S | Supraventricular ectopic (SVEB) |
| 2 | V | Ventricular ectopic (PVC) |
| 3 | F | Fusion beat |
| 4 | Q | Unknown/Paced |

## 📦 Cell 1 — Install & Constants (always run)

In [ ]:
!pip install -q wfdb scikit-learn

import os, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import wfdb
import wfdb.processing as wfdb_proc
from google.colab import drive

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Constants ──────────────────────────────────────────────────────────────
FS          = 360          # MIT-BIH sampling rate
BEAT_LEN    = 187          # samples per beat window
BEAT_BEFORE = 90           # samples before R-peak
BEAT_AFTER  = BEAT_LEN - BEAT_BEFORE
N_CLASSES   = 5
CLASS_NAMES = ['Normal (N)', 'SVEB (S)', 'PVC (V)', 'Fusion (F)', 'Unknown (Q)']
CLASS_SYMS  = ['N', 'S', 'V', 'F', 'Q']

# WFDB annotation symbol → class ID mapping (AAMI standard)
# Multiple symbols map to the same class (e.g. 'A','a','J','S' are all SVEB)
SYM_TO_CLASS = {
    # Normal
    'N':0, 'L':0, 'R':0, 'e':0, 'j':0,
    # SVEB
    'A':1, 'a':1, 'J':1, 'S':1,
    # PVC
    'V':2, 'E':2,
    # Fusion
    'F':3,
    # Unknown
    '/':4, 'f':4, 'Q':4
}

# Records chosen to have good coverage of all 5 classes
RECORD_IDS = [
    '100','103','105','106','108',   # good Normal + some PVC
    '109','111','207',               # LBBB / PVC heavy
    '200','201','202','203',         # SVEB + PVC
    '210','213','215','219',         # mixed
    '221','223','230','231',         # Fusion + varied
]

DATA_DIR     = '/content/mitbih_wfdb'
DRIVE_DIR    = '/content/drive/MyDrive/ECG_Model'
WEIGHTS_PATH = f'{DRIVE_DIR}/ecg_resnet_mitbih.pt'

print(f'Device  : {DEVICE}')
print(f'Records : {len(RECORD_IDS)} MIT-BIH records')
print(f'Weights will be saved → {WEIGHTS_PATH}')

## 🏗️ Cell 2 — Model Architecture (always run)

In [ ]:
class ResBlock(nn.Module):
    """Residual block: Conv → BN → ReLU → Conv → BN + skip connection."""
    def __init__(self, in_ch: int, out_ch: int, downsample: bool = False):
        super().__init__()
        stride     = 2 if downsample else 1
        self.conv1 = nn.Conv1d(in_ch, out_ch, 5, stride=stride, padding=2, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 5, padding=2, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.skip  = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if (in_ch != out_ch or downsample) else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(
            self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.skip(x)
        )


class ECGResNet(nn.Module):
    """
    1D-ResNet for ECG beat classification.
    Input  : (B, 1, 187)  — one normalised beat
    Output : (B, 5)       — logits for 5 AAMI classes

    ⚠️  Do NOT change this class after saving weights.
        Architecture must match exactly when loading.
    """
    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(3, stride=2, padding=1)
        )
        self.layer1 = ResBlock(32,  64)
        self.layer2 = ResBlock(64,  128, downsample=True)
        self.layer3 = ResBlock(128, 256, downsample=True)
        self.pool   = nn.AdaptiveAvgPool1d(1)
        self.drop   = nn.Dropout(0.3)
        self.fc     = nn.Linear(256, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.fc(self.drop(self.pool(x).squeeze(-1)))


print(f'ECGResNet defined — {sum(p.numel() for p in ECGResNet().parameters()):,} parameters')

## 📥 Cell 3 — Download Raw ECG Records from PhysioNet
Uses `wfdb.dl_database` — **official PhysioNet source, always works, no account needed.**

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

def download_record(record_id: str) -> None:
    """Download one MIT-BIH record if not already cached."""
    if os.path.exists(f'{DATA_DIR}/{record_id}.hea'):
        return   # already cached
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR, records=[record_id])

print(f'Downloading {len(RECORD_IDS)} records from PhysioNet...')
for rid in RECORD_IDS:
    download_record(rid)
    print(f'  ✔ {rid}', end='  ', flush=True)
print('\n\nAll records downloaded!')

## 🔬 Cell 4 — Build Dataset: Detect Beats + Label from Annotations

In [ ]:
def normalise(sig: np.ndarray) -> np.ndarray:
    """Z-score normalise a 1-D array."""
    return (sig - sig.mean()) / (sig.std() + 1e-8)


def process_record(record_id: str) -> tuple[np.ndarray, np.ndarray]:
    """
    For one WFDB record:
      1. Load raw signal (MLII lead)
      2. Load ground-truth beat annotations (.atr)
      3. Cut 187-sample window around each annotated beat position
      4. Map annotation symbol → AAMI class ID

    Returns
    -------
    beats  : (M, 187) float32
    labels : (M,)     int  (-1 = unknown symbol, filtered out later)
    """
    path   = f'{DATA_DIR}/{record_id}'
    record = wfdb.rdrecord(path)
    ann    = wfdb.rdann(path, 'atr')
    sig    = record.p_signal[:, 0].astype(np.float32)  # MLII lead
    sig    = normalise(sig)

    beats, labels = [], []
    for pos, sym in zip(ann.sample, ann.symbol):
        cls = SYM_TO_CLASS.get(sym, -1)
        if cls == -1:
            continue                        # skip non-beat annotations
        lo, hi = pos - BEAT_BEFORE, pos + BEAT_AFTER
        if lo < 0 or hi > len(sig):
            continue                        # skip boundary beats
        beat = normalise(sig[lo:hi].copy()) # per-beat normalisation
        beats.append(beat)
        labels.append(cls)

    return np.array(beats, dtype=np.float32), np.array(labels, dtype=int)


def build_dataset(record_ids: list,
                   max_per_class: int = 3000,
                   seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    """
    Process all records → combine → balance classes → return (X, y).
    Using ground-truth .atr annotations for labels (not the detector).
    """
    all_beats, all_labels = [], []

    for rid in record_ids:
        try:
            beats, labels = process_record(rid)
            all_beats.append(beats)
            all_labels.append(labels)
            counts = {CLASS_NAMES[c]: (labels==c).sum() for c in range(N_CLASSES)}
            print(f'  {rid}: {len(beats):4d} beats  {counts}')
        except Exception as e:
            print(f'  {rid}: FAILED — {e}')

    X = np.concatenate(all_beats)
    y = np.concatenate(all_labels)

    # Balance: cap each class at max_per_class
    rng = np.random.default_rng(seed)
    idx = np.concatenate([
        rng.choice(np.where(y==c)[0], min((y==c).sum(), max_per_class), replace=False)
        for c in range(N_CLASSES) if (y==c).sum() > 0
    ])
    rng.shuffle(idx)

    print(f'\nFinal balanced dataset: {len(idx)} beats')
    for c in range(N_CLASSES):
        n = (y[idx]==c).sum()
        print(f'  {CLASS_NAMES[c]:15s}: {n}')
    return X[idx], y[idx]


print('Building dataset from PhysioNet records...')
X, y = build_dataset(RECORD_IDS)

## 🚀 Cell 5 — Train & Save Weights to Google Drive

In [ ]:
# ── Mount Drive ────────────────────────────────────────────────────────────
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

# ── Split ──────────────────────────────────────────────────────────────────
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X,     y,     test_size=0.30, stratify=y,     random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

def make_loader(Xa, ya, shuffle):
    return DataLoader(
        TensorDataset(torch.tensor(Xa[:, np.newaxis, :]),
                      torch.tensor(ya, dtype=torch.long)),
        batch_size=256, shuffle=shuffle
    )

train_loader = make_loader(X_tr,  y_tr,  True)
val_loader   = make_loader(X_val, y_val, False)
test_loader  = make_loader(X_te,  y_te,  False)
print(f'Split — Train: {len(X_tr)}  Val: {len(X_val)}  Test: {len(X_te)}')

# ── Train ──────────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, criterion, train: bool) -> tuple[float, float]:
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if train: optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            n          += len(yb)
    return total_loss / n, correct / n


model     = ECGResNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)
criterion = nn.CrossEntropyLoss()
best_acc  = 0.0

print('\nTraining...\n')
for ep in range(1, 26):
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, criterion, train=True)
    vl_loss, vl_acc = run_epoch(model, val_loader,   optimizer, criterion, train=False)
    scheduler.step()
    tag = ''
    if vl_acc > best_acc:
        best_acc = vl_acc
        torch.save(model.state_dict(), WEIGHTS_PATH)
        tag = '  ✅ saved'
    if ep % 5 == 0 or ep == 1:
        print(f'  Ep {ep:2d}/25  tr_acc={tr_acc:.4f}  val_acc={vl_acc:.4f}{tag}')

print(f'\n🎯 Best val accuracy : {best_acc:.4f}')
print(f'💾 Weights saved to  : {WEIGHTS_PATH}')

## 📊 Cell 6 — Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        all_pred.append(model(xb.to(DEVICE)).argmax(1).cpu())
        all_true.append(yb)
y_pred = torch.cat(all_pred).numpy()
y_true = torch.cat(all_true).numpy()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d',
            cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set(title='Confusion Matrix — Test Set',
       xlabel='Predicted', ylabel='True')
plt.tight_layout(); plt.show()

---
## ✅ Setup Complete!

Your pretrained weights are now permanently saved at:
```
/content/drive/MyDrive/ECG_Model/ecg_resnet_mitbih.pt
```

### Load in your main notebook with just 4 lines:
```python
from google.colab import drive
drive.mount('/content/drive')

model = ECGResNet()   # same architecture class
model.load_state_dict(
    torch.load('/content/drive/MyDrive/ECG_Model/ecg_resnet_mitbih.pt',
               map_location=DEVICE)
)
model.to(DEVICE).eval()
print('✅ Pretrained model loaded — ready for inference')
```

### You NEVER need to re-run this setup notebook.
### The weights stay on your Drive permanently.